# 03 - Classify the GPU catalog

`gpus.csv` alone cannot drive recommendations:

- `type` is missing for 43% of rows and wrong for some (laptop cards tagged Desktop);
- workstation cards hide outside `type=Workstation` (Quadro, Tesla, RTX A, T-series,
  Arc Pro), which is how a T500 could be "recommended";
- there is no price or release year, so "cheapest" means lowest benchmark score.

This notebook writes `curated/gpus_classified.csv` for the serving API:

| column | values |
|---|---|
| `form_factor` | desktop / laptop / integrated / unclassified |
| `market` | consumer / workstation / mining / sample / invalid |
| `availability` | current / aging / legacy_secondhand / unknown / n/a |
| `ff_source` | rule / column / model / unclassified |

Name rules run first; unresolved rows get a TF-IDF + logistic-regression prediction,
accepted only at p >= 0.7. Requires `data/gpus.csv` from `ingest`.


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import HashingTF, IDF, StringIndexer, Tokenizer
from pyspark.ml.functions import vector_to_array

JOBS = os.environ.get('JOBS_DIR', '/home/jovyan/jobs')
if JOBS not in sys.path:
    sys.path.insert(0, JOBS)
from catalog import add_generation
from artifacts import update_registry, write_rows_csv
from schemas import GPUS_SCHEMA

from spark_session import get_spark

DATA = Path(os.environ.get('DATA_DIR', '/home/jovyan/data'))
spark = get_spark('gpu-recsys-catalog')
print('spark', spark.version)

In [ ]:
gpus = spark.read.csv(str(DATA / 'gpus.csv'), header=True, schema=GPUS_SCHEMA)
print(f'rows: {gpus.count()}')
gpus.groupBy('type').count().orderBy(F.desc('count')).show()

## Rule pass

Name rules decide what they can and record how in `ff_source`. Order matters:
integrated tokens beat mobile suffixes, and "Laptop" in the name overrides a wrong
`type`.


In [ ]:
INTEGRATED = r'(?i)uhd graphics|iris|\bhd graphics\b|with radeon graphics|radeon graphics|\bvega (3|6|8|10|11)\b|\badreno\b|apple m\d|radeon \d+m\b|radeon ryzen'
LAPTOP = r'(?i)laptop|mobile|\bmx\d{2,3}\b|\d{3,4}mx\b|\bhs\b|\d+m\b|\d+s\b|\d{4}u\b'
WORKSTATION = r'(?i)quadro|tesla|firepro|\bvgx\b|\brtx a\d{3,4}\b|\brtx pro\b|\barc pro\b|\bradeon pro\b|\bpro w\d|\bpro v\d|\bt\d{3,4}\b'
MINING = r'(?i)\bp10\d-?\d{2,3}\b|\bcmp \d|\bmining\b'
SAMPLE = r'(?i)eng sample|engineering sample|sample:'
VENDOR = r'(?i)geforce|radeon|\brtx\b|\bgtx\b|\bgts\b|\bgt \d|iris|uhd|\barc\b|quadro|tesla|firepro|\bvega\b|adreno|titan|\br[579] \d|\bmx\d|intel|\bnvidia\b|\bamd\b|graphics'

gpus = (
    gpus
    .withColumn('ff_rule',
        F.when(F.col('name').rlike(INTEGRATED), 'integrated')
         .when(F.col('name').rlike(LAPTOP) | (F.col('type') == 'Mobile'), 'laptop')
         .when(F.col('type') == 'Desktop', 'desktop'))
    .withColumn('ff_rule_source',
        F.when(F.col('name').rlike(INTEGRATED), 'rule')
         .when(F.col('name').rlike(LAPTOP) | (F.col('type') == 'Mobile'), 'rule')
         .when(F.col('type') == 'Desktop', 'column'))
    .withColumn('market',
        F.when(F.col('name').rlike(MINING), 'mining')
         .when(F.col('name').rlike(SAMPLE), 'sample')
         .when(F.col('name').rlike(WORKSTATION), 'workstation')
         .when(~F.col('name').rlike(VENDOR), 'invalid')
         .otherwise('consumer'))
)
print('non-retail rows excluded from recommendations:')
gpus.filter(F.col('market').isin('mining', 'sample', 'invalid')).select('name', 'market', 'g3d_mark').show(25, truncate=False)
gpus.groupBy('ff_rule').count().orderBy(F.desc('count')).show()

workstations = gpus.filter(F.col('market') == 'workstation').select('name', 'ff_rule', 'market')
print('worstation cards detected:', workstations.count())
workstations.show(20, truncate=False)

## Classifier for the rest

Trained on the rows the rules already label, scored on the rest, and accepted only
at p >= 0.7. Anything weaker stays `unclassified` and is never recommended.


In [ ]:
labelled = gpus.filter(F.col('ff_rule').isin('desktop', 'laptop'))
train_c, test_c = labelled.randomSplit([0.8, 0.2], seed=3208)

tokenizer = Tokenizer(inputCol='name', outputCol='words')
hashing = HashingTF(inputCol='words', outputCol='tf', numFeatures=4096)
idf = IDF(inputCol='tf', outputCol='tfidf')
label_indexer = StringIndexer(inputCol='ff_rule', outputCol='label')
lr = LogisticRegression(featuresCol='tfidf', labelCol='label', maxIter=50, regParam=0.1)
pipeline = Pipeline(stages=[tokenizer, hashing, idf, label_indexer, lr])

probe = pipeline.fit(train_c)
accuracy = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy').evaluate(probe.transform(test_c))
majority = test_c.groupBy('ff_rule').count().orderBy(F.desc('count')).first()['count'] / test_c.count()
print(f'desktop-vs-laptop holdout accuracy: {accuracy:.3f} (majority-class baseline {majority:.3f}, test n={test_c.count()})')
print('NOTE: the training labels come from the dataset type column, which is itself')
print('unreliable - that is the reason for this notebook - so ~0.8 is the realistic ceiling.')

model = pipeline.fit(labelled)
labels = model.stages[3].labels
print('label indices:', dict(enumerate(labels)))

label_map = F.create_map(*[F.lit(v) for pair in list(enumerate(labels)) for v in pair])
scored = (
    model.transform(gpus)
    .withColumn('confidence', F.element_at(vector_to_array(F.col('probability')), F.col('prediction').cast('int') + 1))
    .withColumn('ff_model', label_map[F.col('prediction').cast('int')])
)

gpus = (
    scored
    .withColumn('form_factor', F.coalesce(F.col('ff_rule'), F.when(F.col('confidence') >= 0.8, F.col('ff_model'))))
    .withColumn('form_factor', F.coalesce(F.col('form_factor'), F.lit('unclassified')))
    .withColumn('ff_source', F.coalesce(F.col('ff_rule_source'), F.when(F.col('confidence') >= 0.8, F.lit('model')).otherwise(F.lit('unclassified'))))
)
gpus.groupBy('ff_source', 'form_factor').count().orderBy(F.desc('count')).show()
gpus.filter(F.col('ff_source') == 'model').select('name', 'form_factor', 'confidence').orderBy(F.desc('confidence')).show(15, truncate=False)

## Age and availability

The data has no release dates, so age comes from a curated generation map
(name -> family -> year). Year sets availability: 2020+ current, 2016-2019 aging,
older legacy_secondhand; integrated GPUs stay `n/a` but still get a generation.
`is_legacy` is 1 before 2020. The map is a heuristic and is audited below.


In [ ]:
from catalog import add_generation

gpus = add_generation(gpus)
gpus.groupBy('availability').count().orderBy(F.desc('count')).show()
gpus.groupBy('generation').count().orderBy(F.desc('count')).show(40, truncate=False)

# audit: anything unmatched with a high score needs a new map entry
gpus.filter((F.col('generation').isNull()) & (F.col('g3d_mark') >= 10000)) \
    .select('name', 'g3d_mark', 'form_factor').orderBy(F.desc('g3d_mark')).show(40, truncate=False)


In [ ]:
OUT = DATA / 'curated'
OUT.mkdir(parents=True, exist_ok=True)
columns = ['name', 'type', 'g3d_mark', 'form_factor', 'market', 'availability',
           'generation', 'gen_year', 'is_legacy', 'ff_source']
rows = gpus.select(*columns).orderBy(F.desc('g3d_mark')).collect()

catalog_path = OUT / 'gpus_classified.csv'
write_rows_csv(catalog_path, columns, rows)

distributions = {
    column: {r[column]: r['count'] for r in gpus.groupBy(column).count().collect()}
    for column in ('form_factor', 'market', 'availability', 'ff_source')
}
generations = {r['generation']: r['count'] for r in gpus.groupBy('generation').count().collect()}
REGISTRY_PATH = OUT / 'model_selection.json'
update_registry(REGISTRY_PATH, {'gpu_catalog': {
    'holdout_accuracy': accuracy,
    'rows': len(rows),
    'distributions': distributions,
    'generations': generations,
}})
print(json.dumps(distributions, indent=2))
print(json.dumps(generations, indent=2, default=str))


## Publish

Upload the classified catalog so the serving API reads generation, year and the
legacy flag at boot.


In [ ]:
from storage import get_bucket, upload_file

upload_file(get_bucket(), 'models/gpus_classified.csv', catalog_path, 'text/csv')
print('uploaded models/gpus_classified.csv')
